In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone - 1

**lOAD AND IMPORT**

In [2]:
import numpy as np
import pandas as pd
import string
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print(train.shape, test.shape)
train.head(3)

(2000, 8) (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


Q 1) **Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?**

In [6]:
freq = train['answer'].value_counts()
print("Frequency distribution:\n", freq)

most_frequent  = freq.max()
least_frequent = freq.min()
result = most_frequent + least_frequent

print(f"\nMost frequent count : {most_frequent}")
print(f"Least frequent count: {least_frequent}")
print(f"Answer (Q1) — Sum   : {result}")

Frequency distribution:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most frequent count : 490
Least frequent count: 324
Answer (Q1) — Sum   : 814


Q 2) **After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?**  

In [8]:
def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

train['cleaned_prompt'] = train['prompt'].apply(clean_text)

all_words = []
for prompt in train['cleaned_prompt']:
    all_words.extend(prompt.split())

vocab = set(all_words)
print(f"Total unique words (vocab size): {len(vocab)}")

Total unique words (vocab size): 859


Q 3) **Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?**

In [9]:
row1_prompt = train.loc[train['id'] == 1, 'cleaned_prompt'].values[0]
words_row1  = row1_prompt.split()

filtered = [w for w in words_row1 if w not in ENGLISH_STOP_WORDS]

print(f"Original word count (Row ID 1) : {len(words_row1)}")
print(f"Answer (Q3) — After stop words : {len(filtered)}")
print(f"Remaining words: {filtered}")

Original word count (Row ID 1) : 22
Answer (Q3) — After stop words : 13
Remaining words: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


Q 4) **Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?**

In [10]:
option_cols = ['A', 'B', 'C', 'D', 'E']

combined_texts = []
for _, row in train.iterrows():
    parts = [str(row['prompt'])] + [str(row[c]) for c in option_cols]
    combined_texts.append(' '.join(parts))

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)

print(f"Answer (Q4) — TF-IDF vocabulary size: {len(vectorizer.vocabulary_)}")

Answer (Q4) — TF-IDF vocabulary size: 2762


Q 5) **Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).**

In [11]:
row1 = train[train['id'] == 1].iloc[0]

prompt_vec  = vectorizer.transform([str(row1['prompt'])])
optionA_vec = vectorizer.transform([str(row1['A'])])

sim = cosine_similarity(prompt_vec, optionA_vec)[0][0]
print(f"Answer (Q5) — Cosine similarity (prompt vs A, Row ID 1): {round(sim, 4)}")

Answer (Q5) — Cosine similarity (prompt vs A, Row ID 1): 0.272


Q 6) **Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.**

In [12]:
correct = 0

for _, row in train.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    
    sims = {}
    for opt in option_cols:
        opt_vec  = vectorizer.transform([str(row[opt])])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
    
    predicted_best = max(sims, key=sims.get)
    if predicted_best == row['answer']:
        correct += 1

pct = (correct / len(train)) * 100
print(f"Correct matches : {correct}/{len(train)}")
print(f"Answer (Q6) — Percentage: {round(pct, 4)}%")

Correct matches : 271/2000
Answer (Q6) — Percentage: 13.55%


Q 7) **If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?**

In [13]:
def map_at_3(ground_truth, predictions):
    score = 0.0
    for k, pred in enumerate(predictions[:3], start=1):
        if pred == ground_truth:
            score = 1.0 / k
            break
    return score

q7 = map_at_3('C', ['C', 'A', 'B'])
print(f"Answer (Q7) — MAP@3 (gt=C, pred=[C,A,B]): {q7}")

Answer (Q7) — MAP@3 (gt=C, pred=[C,A,B]): 1.0


Q 8) **If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?**

In [14]:
q8 = map_at_3('B', ['D', 'B', 'E'])
print(f"Answer (Q8) — MAP@3 (gt=B, pred=[D,B,E]): {q8}")

Answer (Q8) — MAP@3 (gt=B, pred=[D,B,E]): 0.5


Q 9) **The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?**

In [15]:
freq_sorted = train['answer'].value_counts()
top3 = list(freq_sorted.index[:3])
print(f"Top 3 most frequent answers: {top3}")

scores = [map_at_3(row['answer'], top3) for _, row in train.iterrows()]
majority_map3 = np.mean(scores)
print(f"Answer (Q9) — Majority Class Baseline MAP@3: {round(majority_map3, 4)}")

Top 3 most frequent answers: ['B', 'C', 'A']
Answer (Q9) — Majority Class Baseline MAP@3: 0.4212


Q 10) **The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?**

In [16]:
scores_tfidf = []

for _, row in train.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    
    sims = {}
    for opt in option_cols:
        opt_vec   = vectorizer.transform([str(row[opt])])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
    
    ranked = sorted(sims, key=sims.get, reverse=True)[:3]
    scores_tfidf.append(map_at_3(row['answer'], ranked))

tfidf_map3 = np.mean(scores_tfidf)
print(f"Answer (Q10) — TF-IDF Pipeline MAP@3: {round(tfidf_map3, 4)}")

Answer (Q10) — TF-IDF Pipeline MAP@3: 0.2962
